Ingestamos en bronce la cotización de los instrumentos del día. 

Se usa yfinance cómo fuente de información ya qué es una de los proveedores más conocidos y universal lo cuál lo hace ideal para un primer prototipo.
Scope: solo tomamos la data si esta presente en dicha fuente, caso contrario no será ingestada, pero tampoco generará inconvenientes en el pipeline. Esta fuente no incluye información de mercado bonos soberanos de Argentina, qué deberian ser traidos de un proveedor alternativo.

In [0]:
%pip install yfinance

Tabla donde se enlistan todos los tickets, esta tabla se generó en base a todos los simbolos de interés de los cuáles se tiene infomración de transaccciones.

In [0]:
ruta_origen = 'iol_challenge.bronze.tickets'

In [0]:
df = (
        spark.read 
        .format("delta") 
        .table(ruta_origen)
)

Agregamos el sufijo .BA qué es la denominación de yfinance cuando se trabaja con un activo del mercado argentino.

In [0]:
tickets = [row["simbolo_titulo"] + ".BA" for row in df.select("simbolo_titulo").collect()]

In [0]:
tickets[0]

In [0]:
import yfinance as yf

Cargamos la api con una excepción en caso de limit requests. La liberia yf por defecto hace aumaticamente manejo de las excepciones en caso de información no encontrada, limites de requests y nos garantiza qué si hay datos qué no se logran cargar entonces no los obtendremos pero tampoco interrumpirá el proceso.
Agregamos de forma defensiva un bloque try para capturar fallas de la libreria requests en caso de errores de errores HTTP.

In [0]:
import requests

try:
    data_raw = yf.download(
        tickets
        ,start="2026-01-01"
        ,interval="1d"
    )
except requests.exceptions.HTTPError as e:
    if e.response.status_code == 429:
        print("Rate limit alcanzado (HTTP 429). Es necesario aplicar backoff.")
    else:
        print(f"Error HTTP: {e}")
except requests.exceptions.RequestException as e:
    print(f"Error de red o conexión: {e}")

Una vez qué tenemos la información de yfinance cargada tenemos qué llevarla mediante operaciones de pivot y unpivot de spark hacia el formato qué necesitamos, el de una tabla con primary key date e instrumento y con columnas las aperturas, clausuras, máximos, minimos y volumenes transaccionados.
Es una buena practica hacer este procesamiento utilizando spark dataframes ya qué en el caso de qué el volumen de datos escale los issues de performance podrán ser mejor resueltos.

In [0]:
import pyspark.sql.functions as F

In [0]:
data = data_raw.reset_index(level=0)
data.columns = [ (col[0], col[1].replace(".BA", "").replace(".","-")) for col in data.columns]

In [0]:
df = spark.createDataFrame(data)

In [0]:
df.columns

In [0]:
columnas_fijas = ["('Date', '')"]
columnas_variables = [col for col in df.columns if col not in columnas_fijas]

In [0]:
columnas_variables

In [0]:
df_unpivoted = df.unpivot(
    ids=columnas_fijas,
    values=columnas_variables,
    variableColumnName="ticker",
    valueColumnName="value"
)

In [0]:
df_unpivoted.limit(100).display()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import regexp_replace

df_with_tickers = (
    df_unpivoted
    .withColumn("ticker_limpio",  regexp_replace(F.col("ticker"), r"[()'\"]", "") )
    .withColumn("tipo", F.trim(F.split(F.col("ticker_limpio"), ",")[0]))
    .withColumn("simbolo", F.trim(F.split(F.col("ticker_limpio"), ",")[1]))
    .withColumn("date", F.col("('Date', '')").cast("date"))
    .filter(F.col("tipo").isin("Close", "Open","High", "Low", "Volume")) # Solo tomamos F.columnas elementales de la cotizacion
    .filter(F.col("value").isNotNull()) # Solo tomamos tickets con información disponible
    .select(
        F.col("date"), 
        F.col("simbolo"), 
        F.col("tipo"), 
        F.col("value")
    )
)


In [0]:
df_with_tickers.limit(30).display()

In [0]:
df_pivoted = df_with_tickers.groupBy("date", "simbolo").pivot(
    "tipo"
).agg(F.first("value"))

In [0]:
df_pivoted.limit(10).display()

Con la data en el formato de una fila por date y por simbolo completamos un append de la data raw de los instrumentos en el modelo de bronce. Además agregamos timestamp de ejecución, particionamiento y un control de calidad a la información.

In [0]:
target_path = "iol_challenge.bronze.raw_ingestion"

In [0]:
(
    df_pivoted
        .withColumn("data_type", F.lit("instrument"))
        .withColumn("data", F.to_json(F.struct(*df_pivoted.columns), options={"ignoreNullFields": "true"}))
        .withColumn("anio", F.year(F.col("date")))
        .withColumn("mes", F.month(F.col("date")))
        .withColumn("dia", F.month(F.col("date")))
        .withColumn("timestamp_ejecucion", F.now())
        .withColumn("errores_calidad", F.array_remove(
                F.array(
                    F.when(F.col("simbolo").isNull(), "SIMBOLO_NULO"),
                    F.when(F.length(F.col("Close")) < 0, "VALOR_DE_CIERRE_NEGATIVO"),
                    F.when(F.length(F.col("High")) < 0, "VALOR_MAXIMO_NEGATIVO"),
                    F.when(F.length(F.col("Low")) < 0, "VALOR_MINIMO_NEGATIVO"),
                    F.when(F.length(F.col("Open")) < 0, "VALOR_DE_APERTURA_NEGATIVO"),
                    F.when(F.length(F.col("Volume")) < 0, "VOLUMEN_DE_TRANSACCION_NEGATIVO"),
                ),
                None 
            )
        ).withColumn("tiene_errores_calidad",
            F.col("errores_calidad").isNotNull()
        ).select(
            F.col("data_type"),
            F.col("data"),
            F.col("dia"),
            F.col("anio"),
            F.col("mes"),
            F.col("timestamp_ejecucion"),
            F.col("errores_calidad"),
            F.col("tiene_errores_calidad"),
        )
        .write
        .format("delta")      
        .mode("append")
        .partitionBy("anio", "mes", "dia")
        .saveAsTable(target_path)
)

Validamos cuantos datos de cada fuente se han ingestado en bronce

In [0]:
%sql
select count(*), data_type FROM iol_challenge.bronze.raw_ingestion GROUP BY data_type;

Validamos algunos de los datos ingestados

In [0]:
%sql
SELECT * FROM iol_challenge.bronze.raw_ingestion WHERE data_type = 'instrument' LIMIT 10;

In [0]:
%sql
SELECT mes, count(*) FROM iol_challenge.bronze.raw_ingestion GROUP BY mes;
